1. Data Audit

The objective of this phase is to understand the structure and quality
of the training and test datasets before performing EDA, feature
engineering, validation, or model training.

Audit objectives
1. Dataset structure
2. Data types
3. Missing values
4. Cardinality
5. Duplicate observations
6. Train/test consistency
7. Target variable audit

After, EDA starts


In [ ]:
# Import and load data
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sum of Train duplicates", train.duplicated().sum())
train

In [ ]:
# Inspect Columns
print("Train Columns")

for i, col in enumerate(train.columns, start=1):
    print(f"{i:2}. {col}")


print("\nTest Columns")

for i, col in enumerate(test.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# Basic Structure
print("Train Info")
train.info()

print("\nTest Info")
test.info()

In [ ]:
# Data Type Audit
dtype_audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique_values": train.nunique(dropna=True)
})

dtype_audit

In [ ]:
#Missing-value Audit
missing_audit = pd.DataFrame(
    {
    "train_missing": train.isna().sum(),
    "train_missing_pct": (train.isna().mean() * 100).round(2),
    "test_missing": test.isna().sum(),
    "test_missing_pct": (test.isna().mean() * 100).round(2)
    }
)

missing_audit = missing_audit.sort_values(
    "train_missing",
    ascending=False
)

missing_audit

In [ ]:
#Total Missing Values
print("Total missing cells:")

print("Train:", train.isna().sum().sum())
print("Test :", test.isna().sum().sum())

In [ ]:
# Unique Value Audit
cardinality = pd.DataFrame(
    {
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(dropna=True),
    "unique_pct": (train.nunique(dropna=True) / len(train) * 100).round(2)
    }
    )

cardinality = cardinality.sort_values("n_unique", ascending = False)

cardinality

In [ ]:
# Exact duplicates
print("Exact Duplicate Rows:")

print("Train:", train.duplicated().sum())
print("Test :", test.duplicated().sum())

In [ ]:
# Product Store duplicates
pair_columns = ["product_code", "store_code"]

train_pair_duplicates = train.duplicated(subset=pair_columns).sum()

test_pair_duplicates = test.duplicated(
    subset=pair_columns
).sum()

print("Duplicate Product-Store Combinations")

print("Train:", train_pair_duplicates)
print("Test :", test_pair_duplicates)

In [ ]:
# Identify Categorical Columns
categorical_columns = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
# Compare Categorical Domains and discover if categories exist in test but not train
for column in categorical_columns:

    train_values = set(train[column].dropna().unique())
    test_values = set(test[column].dropna().unique())

    test_only_values = test_values - train_values

    print(f"\n{column}")
    print("Train unique values:", len(train_values))
    print("Test unique values :", len(test_values))
    print("Test-only values   :", test_only_values)

In [ ]:
# Target Statistics
target = "total_sales"

print("TARGET:", target)
display(train[target].describe().to_frame().T)

In [ ]:
# Target Validity
print("Missing target values :", train[target].isna().sum())
print("Zero target values    :", (train[target] == 0).sum())
print("Negative target values:", (train[target] < 0).sum())
print("Unique target values  :", train[target].nunique())

In [ ]:
# Target Quantiles
target_quantiles = train[target].quantile(
    [ 0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

target_quantiles

In [ ]:
# Dataset Contract
target = "total_sales"

# Target exists only in train
assert target in train.columns
assert target not in test.columns

# Train and test predictors must match
train_features = set(train.columns) - {target}
test_features = set(test.columns)

assert train_features == test_features

# IDs should be unique
assert train["id"].is_unique
assert test["id"].is_unique

# Target should not contain missing values
assert train[target].notna().all()

print("Target correctly identified.")
print("Train/test predictor columns match.")
print("Train IDs are unique.")
print("Test IDs are unique.")
print("Training target contains no missing values.")
print("\nDataset Contract Passed.")

In [ ]:
# Category integrity audit

category_columns = [
    "fat_content",
    "product_category",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in category_columns:
    print(f"{col}")

    print("\nTrain:")
    display(train[col].value_counts(dropna=False).to_frame("count"))

    print("\nTest:")
    display(test[col].value_counts(dropna=False).to_frame("count"))

In [ ]:
# Check for possible whitespace/case inconsistencies

for col in [
    "fat_content",
    "product_category",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    values = pd.concat(
        [train[col], test[col]],
        ignore_index=True
    ).dropna().astype(str)

    normalized = values.str.strip().str.lower()

    variants = pd.DataFrame({
        "raw": values,
        "normalized": normalized
    }).drop_duplicates()

    grouped = variants.groupby("normalized")["raw"].agg(list)

    inconsistent = grouped[grouped.str.len() > 1]

    print(f"{col}")

    if len(inconsistent) == 0:
        print("No obvious case/whitespace variants found.")
    else:
        display(inconsistent.to_frame("variants"))

In [ ]:
# Product code structure audit

print("Number of unique product codes:")
print(train["product_code"].nunique())

print("\nSample product codes:")
display(train["product_code"].head(20).to_frame())

print("\nProduct code length distribution:")
display(
    train["product_code"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
    .to_frame("count")
)

print("\nProduct code prefix distribution:")
display(
    train["product_code"]
    .astype(str)
    .str[:2]
    .value_counts()
    .head(30)
    .to_frame("count")
)

In [ ]:
# Check whether product codes have a stable relationship with product categories

product_category_mapping = (
    train.groupby("product_code")["product_category"]
    .nunique()
)

print("Product codes associated with more than one raw category:")
print((product_category_mapping > 1).sum())

print("\nMaximum number of raw categories associated with one product code:")
print(product_category_mapping.max())

In [ ]:
# Product-level attribute consistency audit

product_attributes = [
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_category",
    "product_price",
]

for col in product_attributes:
    consistency = train.groupby("product_code")[col].nunique(dropna=False)

    print(col)
    print("Products with exactly one unique value:", (consistency == 1).sum())
    print("Products with more than one unique value:", (consistency > 1).sum())
    print("Maximum unique values for one product:", consistency.max())

In [ ]:
# Number of stores associated with each product

stores_per_product = train.groupby("product_code")["store_code"].nunique()

print("Distribution of number of stores per product:")
display(stores_per_product.value_counts().sort_index().to_frame("number_of_products"))

print("\nMaximum stores observed for one product:", stores_per_product.max())
print("Products appearing in all 10 stores:", (stores_per_product == 10).sum())

# Findings & Inferences

The data audit established the structural properties and quality of the
training and test datasets before any transformation, imputation,
encoding, or feature engineering was applied.

## Key Findings

### 1. Dataset structure

- The training set contains 6,818 observations and 13 columns.
- The test set contains 1,705 observations and 12 columns.
- The predictor columns are consistent between train and test.
- Train and test IDs are unique.
- There are no exact duplicate rows.
- There are no duplicate `product_code + store_code` combinations.
- The target variable, `total_sales`, is present only in the training set
  and contains no missing or invalid values.

### 2. Missing data

Missing values are concentrated in two predictors:

- `product_weight_kg`: approximately 18% missing
- `store_size`: approximately 28% missing

The missingness rates are similar between the training and test sets,
suggesting that missing values are part of the dataset structure rather
than an obvious train/test discrepancy.

The missing values will therefore be investigated further before deciding
how they should be handled.

### 3. Categorical data integrity

`product_category` contains systematic capitalization differences.
For example, the same semantic category appears as:

- `Fruits and Vegetables`
- `fruits and vegetables`
- `FRUITS AND VEGETABLES`

This creates artificial categorical levels. These representations
should be normalized during preprocessing so that semantically identical
categories are treated consistently.

Other major categorical variables, including `fat_content`, `store_code`,
`store_location_tier`, and `store_format`, do not show the same obvious
case/whitespace inconsistency.

### 4. Product-code structure

There are 1,555 unique `product_code` values.

The codes have a fixed length and share the common `PRD-` prefix, while
the remaining characters appear identifier-like. No obvious semantic
meaning can be inferred from the raw character positions during the audit.

Therefore, individual characters should not automatically be treated as
meaningful features. The predictive value of `product_code` itself will
be investigated empirically.

### 5. Product attributes are not uniformly fixed by product code

An important structural observation emerged when checking attributes
within each `product_code`.

Most product codes are associated with multiple observed values of:

- `product_weight_kg`
- `shelf_visibility`
- `product_price`

In contrast, `fat_content` is perfectly consistent within every
`product_code`.

`product_category` also varies for many product codes, but the earlier
category-integrity audit indicates that much of this variation is caused
by capitalization differences.

The variation in weight, visibility, and price suggests that these
variables may capture context beyond immutable product identity,
potentially including product-store or store-specific effects.

This is an inference rather than a confirmed causal interpretation.
The EDA phase will investigate whether these variables systematically
vary with store characteristics and whether product-store interactions
are useful for prediction.

### 6. Product-store coverage

Products do not appear in every store.

The number of stores associated with a product ranges from 1 to 9,
with no product appearing in all 10 stores.

Therefore, the dataset does not form a complete product × store matrix.
This structure will be considered when designing validation strategies
and investigating product-store relationships.

### 7. Target variable

`total_sales` is strictly positive and right-skewed:

- Mean: approximately 2,174.76
- Median: approximately 1,790.89
- Standard deviation: approximately 1,697.89
- Minimum: 32.70
- Maximum: 12,996.82

Because the competition metric is RMSE, large prediction errors in the
upper tail can have a substantial effect on model performance.

A log-transformed target may therefore be worth testing later, but it
will not be assumed to be superior. Any target transformation will be
evaluated using RMSE on the original sales scale.

## Modeling Implications

The audit suggests several questions for the next phases:

1. How strongly do store characteristics relate to sales?
2. How strongly do product characteristics relate to sales?
3. Are price, visibility, and weight systematically associated with
   particular stores or store formats?
4. Can missing `store_size` or `product_weight_kg` values be informed by
   other variables?
5. Does `product_code` provide useful predictive information despite its
   high cardinality?
6. Are product-store interactions important?
7. How should validation account for repeated products across stores?
8. Do nonlinear relationships justify tree-based boosting models?
9. Does transforming the target improve RMSE on the original sales scale?

These questions will guide the exploratory analysis, validation design,
feature engineering, and model experiments that follow.

> **Conclusion:** The dataset is structurally consistent between
> train and test, but contains meaningful missingness, high-cardinality
> product identifiers, categorical inconsistencies, and substantial
> variation in several product attributes. These characteristics warrant
> investigation before modeling rather than being handled with generic
> preprocessing assumptions.

# Exploratory Data Analysis

The data audit established the structure, quality, and consistency of the
training and test datasets without modifying the raw data.

This phase investigates the statistical distributions and relationships
within the data to identify patterns that can inform validation, feature
engineering, and model selection.

The analysis is hypothesis-driven. Each investigation should provide
evidence for or against a modeling assumption.

### EDA objectives

1. Understand the distribution of the target variable.
2. Investigate numerical feature distributions and relationships.
3. Examine categorical feature effects and group differences.
4. Investigate missing-value patterns.
5. Explore product-level and store-level structure.
6. Identify potential nonlinearities and interactions.
7. Generate statistically motivated feature-engineering hypotheses.

No feature engineering or model fitting will be performed until the
relevant exploratory evidence has been examined.

In [ ]:
# Target variable: descriptive statistics

target = train["total_sales"]

target_stats = target.describe(percentiles=[
    0.01, 0.05, 0.10,
    0.25, 0.50, 0.75,
    0.90, 0.95, 0.99
])

target_stats

In [ ]:
# Additional target statistics

target_summary = pd.Series({
    "mean": target.mean(),
    "median": target.median(),
    "std": target.std(),
    "variance": target.var(),
    "min": target.min(),
    "max": target.max(),
    "skewness": target.skew(),
    "kurtosis": target.kurt(),
    "iqr": target.quantile(0.75) - target.quantile(0.25),
    "coefficient_of_variation": target.std() / target.mean(),
    "unique_values": target.nunique()
})

target_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    target,
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Total Sales")
plt.xlabel("Total Sales")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig("../Gallery/target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Log-transformed target distribution

log_target = np.log1p(target)

plt.figure(figsize=(10, 6))

plt.hist(
    log_target,
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Log-Transformed Total Sales")
plt.xlabel("log1p(Total Sales)")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig("../Gallery/log_target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compare raw and log-transformed target distributions statistically

target_distribution_comparison = pd.DataFrame({
    "Raw total_sales": [
        target.mean(),
        target.std(),
        target.skew(),
        target.kurt(),
        target.quantile(0.01),
        target.quantile(0.50),
        target.quantile(0.99)
    ],
    "Log1p total_sales": [
        log_target.mean(),
        log_target.std(),
        log_target.skew(),
        log_target.kurt(),
        log_target.quantile(0.01),
        log_target.quantile(0.50),
        log_target.quantile(0.99)
    ]
}, index=[
    "Mean",
    "Std",
    "Skewness",
    "Kurtosis",
    "1st percentile",
    "Median",
    "99th percentile"
])

target_distribution_comparison

In [ ]:
# Target concentration across sales quantiles

target_bins = pd.qcut(
    target,
    q=[0, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00],
    duplicates="drop"
)

target_tail_summary = (
    train.groupby(target_bins, observed=True)["total_sales"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

target_tail_summary

In [ ]:
# Contribution of each target region to total observed sales

target_mass = (
    train.groupby(target_bins, observed=True)["total_sales"]
    .agg(
        count="size",
        total_sales="sum"
    )
)

target_mass["observation_share"] = (
    target_mass["count"] / len(train)
)

target_mass["sales_share"] = (
    target_mass["total_sales"] / target.sum()
)

target_mass

In [ ]:
# Prepare target-mass data for plotting

plot_data = (
    target_mass
    .rename_axis("target_group")
    .reset_index()
)

labels = [
    "Bottom 50%",
    "50-75%",
    "75-90%",
    "90-95%",
    "95-99%",
    "Top 1%"
]

plot_data["target_group_label"] = labels

plt.figure(figsize=(10, 6))

plt.bar(
    plot_data["target_group_label"],
    plot_data["sales_share"] * 100,
    edgecolor="black"
)

plt.title("Share of Total Sales Across Target Quantile Groups")
plt.xlabel("Target quantile group")
plt.ylabel("Share of total sales (%)")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    "../Gallery/target_sales_mass_by_quantile.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

## Target Variable Findings

The target variable `total_sales` exhibits substantial positive
skewness and dispersion.

- Mean sales are approximately 2,174.76, compared with a median of
  approximately 1,790.89.
- The raw target has skewness of approximately 1.15, confirming a
  pronounced right-skewed distribution.
- The standard deviation is approximately 1,697.89, giving a
  coefficient of variation of approximately 0.78.
- The upper tail extends to approximately 12,996.82, although only
  about 1% of observations exceed 7,261.
- The upper half of observations accounts for approximately 80% of
  the total observed sales value, while the lower half accounts for
  approximately 20%.
- The top 10% of observations account for approximately 26.8% of
  total observed sales.

Applying `log1p` substantially reduces the original positive
skewness, but produces moderate negative skewness rather than a
symmetric distribution. Therefore, the log transformation should not
be assumed to be superior based on distributional shape alone.

### Modeling implications

The target distribution motivates several hypotheses for later
experiments:

1. Raw-target and log-target models should both be evaluated.
2. Evaluation must remain on the original sales scale using RMSE.
3. Model residuals should later be examined across target quantiles.
4. Particular attention should be given to systematic
   underprediction or overprediction in the upper tail.
5. Target transformation should be treated as an empirical modeling
   decision rather than a preprocessing requirement.

In [ ]:
# Statistical profile of product_price

price = train["product_price"]

price_summary = pd.Series({
    "count": price.count(),
    "mean": price.mean(),
    "median": price.median(),
    "std": price.std(),
    "variance": price.var(),
    "min": price.min(),
    "1%": price.quantile(0.01),
    "5%": price.quantile(0.05),
    "25%": price.quantile(0.25),
    "50%": price.quantile(0.50),
    "75%": price.quantile(0.75),
    "95%": price.quantile(0.95),
    "99%": price.quantile(0.99),
    "max": price.max(),
    "skewness": price.skew(),
    "kurtosis": price.kurt(),
    "iqr": price.quantile(0.75) - price.quantile(0.25),
    "coefficient_of_variation": price.std() / price.mean(),
    "unique_values": price.nunique()
})

price_summary

In [ ]:
# Distribution of product_price

plt.figure(figsize=(10, 6))

plt.hist(
    train["product_price"],
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Product Price")
plt.xlabel("Product price")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "../Gallery/product_price_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Relationship between product price and total sales
# using price quantile bins

price_sales = train[["product_price", "total_sales"]].copy()

price_sales["price_quantile"] = pd.qcut(
    price_sales["product_price"],
    q=10,
    duplicates="drop"
)

price_sales_summary = (
    price_sales
    .groupby("price_quantile", observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        median_price=("product_price", "median"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std")
    )
)

price_sales_summary

In [ ]:
# Correlation between product price and total sales

price_sales_corr = pd.Series({
    "Pearson": train["product_price"].corr(
        train["total_sales"],
        method="pearson"
    ),
    "Spearman": train["product_price"].corr(
        train["total_sales"],
        method="spearman"
    )
})

price_sales_corr

In [ ]:
# Product price vs total sales

plt.figure(figsize=(10, 6))

plt.scatter(
    train["product_price"],
    train["total_sales"],
    edgecolor = "black",
    alpha=0.8,
    s=18
)

plt.title("Product Price vs Total Sales")
plt.xlabel("Product price")
plt.ylabel("Total sales")

plt.tight_layout()

plt.savefig(
    "../Gallery/product_price_vs_total_sales.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Product category sales profile

category_sales = (
    train
    .groupby("product_category")["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .sort_values("mean_sales", ascending=False)
)

category_sales

In [ ]:
# Normalize product category casing

train["product_category_clean"] = train["product_category"].str.strip().str.lower()
test["product_category_clean"] = test["product_category"].str.strip().str.lower()

print("Unique raw categories  :", train["product_category"].nunique())
print("Unique cleaned categories:", train["product_category_clean"].nunique())

print("\nCleaned categories:")
print(sorted(train["product_category_clean"].unique()))

In [ ]:
category_sales_clean = (
    train
    .groupby("product_category_clean")["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .sort_values("mean_sales", ascending=False)
)

category_sales_clean

In [ ]:
category_price_corr = (
    train
    .groupby("product_category_clean")
    .agg(
        observations=("total_sales", "size"),
        pearson_corr=("product_price", lambda x: x.corr(
            train.loc[x.index, "total_sales"], method="pearson"
        )),
        spearman_corr=("product_price", lambda x: x.corr(
            train.loc[x.index, "total_sales"], method="spearman"
        ))
    )
    .sort_values("pearson_corr", ascending=False)
)

category_price_corr

In [ ]:
price_category_bins = (
    train
    .assign(
        price_bin=pd.qcut(
            train["product_price"],
            q=5,
            duplicates="drop"
        )
    )
    .groupby(
        ["product_category_clean", "price_bin"],
        observed=True
    )
    .agg(
        observations=("total_sales", "size"),
        median_sales=("total_sales", "median"),
        mean_sales=("total_sales", "mean")
    )
    .reset_index()
)

price_category_bins.head(20)

In [ ]:
# Compare median sales across global price bins for every product category
price_category_median = (
    price_category_bins
    .pivot(
        index="product_category_clean",
        columns="price_bin",
        values="median_sales"
    )
)

price_category_median

In [ ]:
# Identify category × price-bin groups with fewer than 20 observations
small_price_category_groups = (
    price_category_bins
    .loc[
        price_category_bins["observations"] < 20,
        ["product_category_clean", "price_bin", "observations", "median_sales", "mean_sales"]
    ]
    .sort_values(["product_category_clean", "observations"])
)

small_price_category_groups

In [ ]:
# Visualize median sales across global price bins for each product category
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(16, 14), sharex=True, sharey=True)

for ax, category in zip(axes.flat, price_category_median.index):
    values = price_category_median.loc[category]
    ax.plot(range(1, len(values) + 1), values.values, marker="o")
    ax.set_title(category.title())
    ax.set_xticks(range(1, len(values) + 1))
    ax.set_xticklabels(["Q1", "Q2", "Q3", "Q4", "Q5"])
    ax.grid(alpha=0.3)

fig.supxlabel("Global Product Price Quintile")
fig.supylabel("Median Total Sales")
fig.suptitle("Median Sales Across Product Price Quintiles by Category", y=1.02)
plt.tight_layout()

plt.savefig("../Gallery/price_category_median_sales.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Programmatically validate the main visual claims from the category price-quintile plot
plot_validation = price_category_median.copy()

plot_validation["Q1_to_Q5_change"] = plot_validation.iloc[:, -1] - plot_validation.iloc[:, 0]
plot_validation["Q1_to_Q5_ratio"] = plot_validation.iloc[:, -1] / plot_validation.iloc[:, 0]

plot_validation["monotonic_increase"] = (
    plot_validation.iloc[:, :5]
    .diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all(axis=1)
)

plot_validation["largest_jump"] = (
    plot_validation.iloc[:, :5]
    .diff(axis=1)
    .iloc[:, 1:]
    .idxmax(axis=1)
)

plot_validation[
    [
        "Q1_to_Q5_change",
        "Q1_to_Q5_ratio",
        "monotonic_increase",
        "largest_jump"
    ]
].sort_values("Q1_to_Q5_change", ascending=False)

In [ ]:
# Compare total_sales distributions across individual stores using summary statistics
store_sales_profile = (
    train
    .groupby("store_code")
    .agg(
        observations=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std"),
        min_sales=("total_sales", "min"),
        max_sales=("total_sales", "max")
    )
    .sort_values("mean_sales", ascending=False)
)

store_sales_profile

In [ ]:
# Show the distribution of store formats within each store code
store_format_by_code = (
    pd.crosstab(
        train["store_code"],
        train["store_format"],
        normalize="index"
    )
    .mul(100)
    .round(2)
)

store_format_by_code

In [ ]:
store_structure = (
    train
    .groupby("store_code")
    .agg(
        observations=("store_code", "size"),
        formats=("store_format", "nunique"),
        sizes=("store_size", "nunique"),
        location_tiers=("store_location_tier", "nunique"),
        ages=("store_age_years", "nunique")
    )
)

store_structure

In [ ]:
store_profile = (
    train
    .groupby("store_code")
    .agg(
        observations=("store_code", "size"),
        store_format=("store_format", "first"),
        store_size=("store_size", lambda x: x.dropna().unique().tolist()),
        store_location_tier=("store_location_tier", "first"),
        store_age_years=("store_age_years", "first")
    )
)

store_profile

In [ ]:
format_sales_profile = (
    train
    .groupby("store_format")["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .sort_values("mean_sales", ascending=False)
)

format_sales_profile

In [ ]:
size_sales_profile = (
    train
    .groupby("store_size", dropna=False)["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .sort_values("mean_sales", ascending=False)
)

size_sales_profile

In [ ]:
# Summarize total_sales across store location tiers, including any missing tier values.
tier_sales_profile = (
    train
    .groupby("store_location_tier", dropna=False)["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .sort_values("mean_sales", ascending=False)
)

tier_sales_profile

In [ ]:
# Create weight quantile bins and explicitly allow a separate Missing category.
weight_sales = train[["product_weight_kg", "total_sales"]].copy()

weight_sales["weight_bin"] = pd.qcut(
    weight_sales["product_weight_kg"],
    q=10,
    duplicates="drop"
)

weight_sales["weight_bin"] = (
    weight_sales["weight_bin"]
    .cat.add_categories("Missing")
)

weight_sales.loc[
    weight_sales["product_weight_kg"].isna(),
    "weight_bin"
] = "Missing"

weight_sales_profile = (
    weight_sales
    .groupby("weight_bin", observed=False)["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
)

weight_sales_profile

In [ ]:
# Compare product-weight missingness across normalized product categories.
weight_missing_by_category = (
    train
    .assign(weight_missing=train["product_weight_kg"].isna())
    .groupby("product_category_clean")["weight_missing"]
    .agg(
        observations="size",
        missing_count="sum",
        missing_rate="mean"
    )
    .sort_values("missing_rate", ascending=False)
)

weight_missing_by_category

In [ ]:
# Classify each product_code by how consistently its product weight is missing.
weight_missing_by_product = (
    train
    .groupby("product_code")["product_weight_kg"]
    .agg(
        observations="size",
        missing_count=lambda x: x.isna().sum()
    )
)

weight_missing_by_product["missing_rate"] = (
    weight_missing_by_product["missing_count"]
    / weight_missing_by_product["observations"]
)

weight_missing_pattern = (
    weight_missing_by_product["missing_rate"]
    .map(
        lambda x: (
            "none_missing" if x == 0
            else "all_missing" if x == 1
            else "partially_missing"
        )
    )
    .value_counts()
    .rename_axis("missing_pattern")
    .to_frame("product_count")
)

weight_missing_pattern

In [ ]:
# Measure product-weight missingness within each store to see whether missingness has a store-level pattern.
weight_missing_by_store = (
    train
    .assign(weight_missing=train["product_weight_kg"].isna())
    .groupby("store_code")["weight_missing"]
    .agg(
        observations="size",
        missing_count="sum",
        missing_rate="mean"
    )
    .sort_values("missing_rate", ascending=False)
)

weight_missing_by_store

In [ ]:
# Compare observed product-weight distributions across stores using count, median, and spread.
weight_by_store = (
    train
    .groupby("store_code")["product_weight_kg"]
    .agg(
        observations="count",
        missing_count=lambda x: x.isna().sum(),
        median_weight="median",
        mean_weight="mean",
        std_weight="std",
        min_weight="min",
        max_weight="max"
    )
)

weight_by_store

In [ ]:
# Measure how much product_weight_kg varies within each product_code across its observed records.
weight_variation_by_product = (
    train
    .groupby("product_code")["product_weight_kg"]
    .agg(
        observations="size",
        observed_values="count",
        unique_weights="nunique",
        median_weight="median",
        min_weight="min",
        max_weight="max"
    )
)

weight_variation_by_product["weight_range"] = (
    weight_variation_by_product["max_weight"]
    - weight_variation_by_product["min_weight"]
)

weight_variation_by_product.describe()

In [ ]:
# Count observed product-weight values available for products with partial missingness.
partial_weight_products = (
    weight_variation_by_product
    .loc[
        (weight_variation_by_product["observed_values"] > 0) &
        (weight_variation_by_product["observed_values"] < weight_variation_by_product["observations"])
    ]
)

partial_weight_products["observed_values"].value_counts().sort_index()


In [ ]:
# Compare within-product weight variation across different numbers of observed weight values.
weight_stability_by_count = (
    partial_weight_products
    .groupby("observed_values")["weight_range"]
    .agg(
        products="count",
        median_range="median",
        mean_range="mean",
        max_range="max"
    )
)

weight_stability_by_count

In [ ]:
# Inspect the eight products whose product weight is completely missing in the training data.
all_missing_weight_products = weight_missing_by_product.loc[
    weight_missing_by_product["missing_rate"] == 1
]

all_missing_weight_products

In [ ]:
visibility = train["shelf_visibility"]

visibility_summary = pd.Series({
    "count": visibility.count(),
    "missing": visibility.isna().sum(),
    "unique": visibility.nunique(),
    "mean": visibility.mean(),
    "median": visibility.median(),
    "std": visibility.std(),
    "min": visibility.min(),
    "max": visibility.max(),
    "skewness": visibility.skew(),
    "kurtosis": visibility.kurtosis(),
    "zeros": (visibility == 0).sum(),
    "zero_pct": (visibility == 0).mean() * 100
})

visibility_quantiles = visibility.quantile(
    [0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

display(visibility_summary.to_frame("value"))
display(visibility_quantiles.to_frame("quantile"))

In [ ]:
visibility_sales_corr = pd.Series({
    "pearson": train["shelf_visibility"].corr(train["total_sales"], method="pearson"),
    "spearman": train["shelf_visibility"].corr(train["total_sales"], method="spearman")
})

display(visibility_sales_corr.to_frame("correlation"))


import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = train.select_dtypes(include=["number"])
corr_matrix = numeric_cols.corr(method="pearson")
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_matrix,
    annot=True,  
)

plt.title("Numeric Columns Correlation Heatmap")
plt.savefig("../Gallery/Numeric Columns Correlation Heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Create visibility bins and summarize total_sales across them to check for nonlinear or regime-specific patterns.
visibility_sales = train[["shelf_visibility", "total_sales"]].copy()

visibility_sales["visibility_bin"] = pd.qcut(
    visibility_sales["shelf_visibility"],
    q=10,
    duplicates="drop"
)

visibility_sales_profile = (
    visibility_sales
    .groupby("visibility_bin", observed=True)["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max"
    )
    .reset_index()
)

display(visibility_sales_profile)

In [ ]:
# Compare total_sales across within-category shelf-visibility quartiles to test whether the high-visibility pattern is consistent across product categories.
visibility_category = train[
    ["product_category_clean", "shelf_visibility", "total_sales"]
].copy()

visibility_category["visibility_quartile"] = (
    visibility_category
    .groupby("product_category_clean")["shelf_visibility"]
    .transform(
        lambda x: pd.qcut(x, q=4, labels=False, duplicates="drop") + 1
    )
)

visibility_category_profile = (
    visibility_category
    .groupby(
        ["product_category_clean", "visibility_quartile"],
        observed=True
    )["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        std_sales="std"
    )
    .reset_index()
)

display(visibility_category_profile)

In [ ]:
# Audit fat_content and summarize total_sales by fat-content group.
fat_content_summary = train.groupby("fat_content")["total_sales"].agg(
    observations="count",
    mean_sales="mean",
    median_sales="median",
    std_sales="std",
    min_sales="min",
    max_sales="max"
).round(2)

print("Value counts:")
print(train["fat_content"].value_counts(dropna=False))

print("\nSales profile by fat_content:")
display(fat_content_summary)

In [ ]:
# Compare mean total_sales for Low Fat and Regular products within each cleaned product category.
fat_category_sales = (
    train.groupby(["product_category_clean", "fat_content"])["total_sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median"
    )
    .round(2)
    .reset_index()
)

display(fat_category_sales)

In [ ]:
# Measure how fat_content is distributed within each cleaned product category.
fat_category_distribution = pd.crosstab(
    train["product_category_clean"],
    train["fat_content"],
    normalize="index"
).round(3)

display(fat_category_distribution)

In [ ]:
# Aim: Audit the distribution, missingness, and basic statistical properties of product_weight_kg.
weight_summary = train["product_weight_kg"].describe()

weight_missing = train["product_weight_kg"].isna().sum()
weight_missing_pct = train["product_weight_kg"].isna().mean() * 100
weight_unique = train["product_weight_kg"].nunique()

weight_skew = train["product_weight_kg"].skew()
weight_kurtosis = train["product_weight_kg"].kurtosis()

weight_quantiles = train["product_weight_kg"].quantile(
    [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

print("Summary statistics:")
display(weight_summary)

print(f"\nMissing values: {weight_missing:,} ({weight_missing_pct:.2f}%)")
print(f"Unique non-missing values: {weight_unique:,}")
print(f"Skewness: {weight_skew:.4f}")
print(f"Kurtosis: {weight_kurtosis:.4f}")

print("\nSelected quantiles:")
display(weight_quantiles)

In [ ]:
# Aim: Measure the linear and monotonic association between product weight and total sales.
weight_sales = train[["product_weight_kg", "total_sales"]].dropna()

weight_pearson = weight_sales["product_weight_kg"].corr(
    weight_sales["total_sales"],
    method="pearson"
)

weight_spearman = weight_sales["product_weight_kg"].corr(
    weight_sales["total_sales"],
    method="spearman"
)

print(f"Pearson correlation:  {weight_pearson:.4f}")
print(f"Spearman correlation: {weight_spearman:.4f}")

In [ ]:
# Check whether the weak overall weight-sales relationship changes across product categories.
weight_category_sales = (
    train.dropna(subset=["product_weight_kg"])
    .groupby("product_category_clean")
    .agg(
        n=("product_weight_kg", "size"),
        mean_weight=("product_weight_kg", "mean"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        weight_sales_corr=(
            "product_weight_kg",
            lambda x: x.corr(
                train.loc[x.index, "total_sales"],
                method="spearman"
            )
        )
    )
    .sort_values("mean_sales", ascending=False)
)

display(weight_category_sales)

In [ ]:
# Audit the distribution and basic statistical properties of store_age_years.
store_age_summary = train["store_age_years"].describe()

store_age_missing = train["store_age_years"].isna().sum()
store_age_missing_pct = train["store_age_years"].isna().mean() * 100
store_age_unique = train["store_age_years"].nunique()

store_age_skew = train["store_age_years"].skew()
store_age_kurtosis = train["store_age_years"].kurtosis()

store_age_quantiles = train["store_age_years"].quantile(
    [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

print("Summary statistics:")
display(store_age_summary)

print(f"\nMissing values: {store_age_missing:,} ({store_age_missing_pct:.2f}%)")
print(f"Unique values: {store_age_unique:,}")
print(f"Skewness: {store_age_skew:.4f}")
print(f"Kurtosis: {store_age_kurtosis:.4f}")

print("\nSelected quantiles:")
display(store_age_quantiles)

In [ ]:
# Examine sales behavior across the observed store_age_years values.
store_age_sales = (
    train.groupby("store_age_years")
    .agg(
        n=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std")
    )
    .reset_index()
    .sort_values("store_age_years")
)

display(store_age_sales)

In [ ]:
# Examine whether store_format composition differs substantially across store_age_years.
store_age_format = (
    train.groupby(["store_age_years", "store_format"])
    .size()
    .unstack(fill_value=0)
)

display(store_age_format)

In [ ]:
# Check whether each store_code is associated with a single store_age_years value.
store_age_code = (
    train.groupby("store_code")["store_age_years"]
    .nunique()
    .value_counts()
    .sort_index()
)

display(store_age_code)

In [ ]:
price_category_analysis = (
    train.groupby("product_category_clean")[[
        "product_price",
        "total_sales"
    ]]
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "price_mean": g["product_price"].mean(),
            "price_median": g["product_price"].median(),
            "sales_mean": g["total_sales"].mean(),
            "sales_median": g["total_sales"].median(),
            "pearson_price_sales": g["product_price"].corr(
                g["total_sales"], method="pearson"
            ),
            "spearman_price_sales": g["product_price"].corr(
                g["total_sales"], method="spearman"
            ),
        })
    )
    .reset_index()
    .sort_values("n", ascending=False)
)

price_category_analysis

In [ ]:
# Store x Product Price
store_price_analysis = (
    train.groupby("store_code")[[
        "product_price",
        "total_sales"
    ]]
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "price_mean": g["product_price"].mean(),
            "price_median": g["product_price"].median(),
            "sales_mean": g["total_sales"].mean(),
            "sales_median": g["total_sales"].median(),
            "pearson_price_sales": g["product_price"].corr(
                g["total_sales"], method="pearson"
            ),
            "spearman_price_sales": g["product_price"].corr(
                g["total_sales"], method="spearman"
            ),
        })
    )
    .reset_index()
    .sort_values("sales_mean", ascending=False)
)

store_price_analysis

In [ ]:
# Examine category sales within each store using mean sales and sample size together
store_category_analysis = (
    train.groupby(["store_code", "product_category_clean"])
    .agg(
        n=("total_sales", "size"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
    )
    .reset_index()
)

store_category_analysis.sort_values(
    ["store_code", "mean_sales"],
    ascending=[True, False]
).head(30)

In [ ]:
# Measure of how much category composition differs between stores
store_category_mix = pd.crosstab(
    train["store_code"],
    train["product_category_clean"],
    normalize="index"
) * 100

store_category_mix.round(2)

In [ ]:
weight_price_corr = (
    train.loc[
        train["product_weight_kg"].notna(),
        ["product_weight_kg", "product_price", "total_sales"]
    ]
    .corr(method="pearson")
)

weight_price_corr.round(4)

In [ ]:
weight_price_analysis = (
    train.loc[train["product_weight_kg"].notna()].copy()
)

weight_price_analysis["price_per_kg"] = (
    weight_price_analysis["product_price"]
    / weight_price_analysis["product_weight_kg"]
)

weight_price_analysis[
    ["price_per_kg", "product_price", "product_weight_kg", "total_sales"]
].describe().round(3)

In [ ]:
ratio_corr = (
    weight_price_analysis[
        ["product_price", "product_weight_kg", "price_per_kg", "total_sales"]
    ]
    .corr(method="spearman")
)

ratio_corr.round(4)

In [ ]:
visibility_analysis = train[
    [
        "shelf_visibility",
        "product_price",
        "total_sales",
        "store_code"
    ]
].copy()

visibility_analysis[
    ["shelf_visibility", "product_price", "total_sales"]
].corr(method="pearson").round(4)

In [ ]:
visibility_analysis[
    ["shelf_visibility", "product_price", "total_sales"]
].corr(method="spearman").round(4)

In [ ]:
price_category_analysis = (
    train.groupby("product_category_clean")
    .apply(
        lambda g: pd.Series({
            "n": g["product_price"].notna().sum(),
            "price_mean": g["product_price"].mean(),
            "price_median": g["product_price"].median(),
            "sales_mean": g["total_sales"].mean(),
            "sales_median": g["total_sales"].median(),
            "pearson_price_sales": g[["product_price", "total_sales"]].corr().iloc[0, 1],
            "spearman_price_sales": g[["product_price", "total_sales"]].corr(method="spearman").iloc[0, 1],
        })
    )
    .sort_values("pearson_price_sales", ascending=False)
)

price_category_analysis.round(4)

In [ ]:
price_variation_by_product = (
    train.groupby("product_code")["product_price"]
    .agg(
        observations="count",
        unique_prices="nunique",
        median_price="median",
        min_price="min",
        max_price="max",
        price_range=lambda x: x.max() - x.min()
    )
)

price_variation_by_product["price_range_pct"] = (
    price_variation_by_product["price_range"]
    / price_variation_by_product["median_price"]
    * 100
)

price_variation_by_product.describe()

In [ ]:
# Visualize the distribution of relative price dispersion across product groups.
import matplotlib.pyplot as plt

median_dispersion = price_variation_by_product["price_range_pct"].median()

plt.figure(figsize=(10, 6))

plt.hist(
    price_variation_by_product["price_range_pct"],
    bins=30,
    edgecolor="black",
    alpha=0.8
)

plt.axvline(
    median_dispersion,
    linestyle="--",
    linewidth=2,
    label=f"Median = {median_dispersion:.2f}%"
)

plt.xlabel("Relative price range (%)")
plt.ylabel("Number of product groups")
plt.title("Distribution of Relative Price Dispersion Across Products")
plt.legend()

plt.tight_layout()
plt.savefig(
    "../Gallery/product_price_dispersion_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Test whether relative price dispersion increases simply because products have more observations.
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    price_variation_by_product["observations"],
    price_variation_by_product["price_range_pct"],
    alpha=0.55,
    edgecolor = "black",
    s=35
)

plt.xlabel("Number of observations per product")
plt.ylabel("Relative price range (%)")
plt.title("Price Dispersion vs Number of Product Observations")

plt.tight_layout()
plt.savefig(
    "../Gallery/price_dispersion_vs_observations.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Identify the products with the highest within-product relative price dispersion.
top_price_dispersion = (
    price_variation_by_product
    .sort_values("price_range_pct", ascending=False)
    .head(15)
    .copy()
)

top_price_dispersion

In [ ]:
# Inspect the highest price-dispersion products across their observed stores and product attributes.
top_price_products = top_price_dispersion.index

top_price_rows = (
    train[train["product_code"].isin(top_price_products)]
    .sort_values(["product_code", "product_price"])
    [[
        "product_code",
        "product_category",
        "product_price",
        "store_code",
        "store_location_tier",
        "store_format",
        "store_size",
        "store_age_years",
        "product_weight_kg",
        "shelf_visibility",
        "total_sales"
    ]]
    .copy()
)

top_price_rows

In [ ]:
# Measure each store's typical price deviation from the corresponding product's median price.
product_median_price = (
    train.groupby("product_code")["product_price"]
    .median()
    .rename("product_median_price")
)

store_price_deviation = train[["product_code", "store_code", "product_price"]].copy()

store_price_deviation = store_price_deviation.join(
    product_median_price,
    on="product_code"
)

store_price_deviation["price_deviation"] = (
    store_price_deviation["product_price"]
    - store_price_deviation["product_median_price"]
)

store_price_deviation["price_deviation_pct"] = (
    store_price_deviation["price_deviation"]
    / store_price_deviation["product_median_price"]
    * 100
)

store_price_deviation_profile = (
    store_price_deviation
    .groupby("store_code")
    .agg(
        observations=("price_deviation_pct", "size"),
        mean_deviation_pct=("price_deviation_pct", "mean"),
        median_deviation_pct=("price_deviation_pct", "median"),
        std_deviation_pct=("price_deviation_pct", "std"),
        mean_abs_deviation_pct=("price_deviation_pct", lambda x: x.abs().mean())
    )
    .sort_values("median_deviation_pct")
)

store_price_deviation_profile

In [ ]:
# Test whether product-relative price deviations differ systematically by store format.
store_format_price_deviation = store_price_deviation.merge(
    train[["product_code", "store_code", "store_format"]].drop_duplicates(),
    on=["product_code", "store_code"],
    how="left"
)

store_format_price_deviation_profile = (
    store_format_price_deviation
    .groupby("store_format")
    .agg(
        observations=("price_deviation_pct", "size"),
        mean_deviation_pct=("price_deviation_pct", "mean"),
        median_deviation_pct=("price_deviation_pct", "median"),
        std_deviation_pct=("price_deviation_pct", "std"),
        mean_abs_deviation_pct=("price_deviation_pct", lambda x: x.abs().mean())
    )
    .sort_values("median_deviation_pct")
)

store_format_price_deviation_profile

In [ ]:
# Inspect the existing weight-missingness DataFrames
print("weight_missing_pattern:")
print(weight_missing_pattern.columns.tolist())
display(weight_missing_pattern)

print("\nweight_missing_by_category:")
print(weight_missing_by_category.columns.tolist())
display(weight_missing_by_category.head(20))

print("\nweight_missing_by_store:")
print(weight_missing_by_store.columns.tolist())
display(weight_missing_by_store.head(20))

print("\nweight_missing_by_product:")
print(weight_missing_by_product.columns.tolist())
display(weight_missing_by_product.head(20))

In [ ]:
# Compare product-weight missingness by store between train and test to verify structural consistency.
train_weight_missing_by_store = (
    train.groupby("store_code")["product_weight_kg"]
    .agg(
        observations="size",
        missing_count=lambda x: x.isna().sum(),
        missing_rate=lambda x: x.isna().mean()
    )
)

test_weight_missing_by_store = (
    test.groupby("store_code")["product_weight_kg"]
    .agg(
        observations="size",
        missing_count=lambda x: x.isna().sum(),
        missing_rate=lambda x: x.isna().mean()
    )
)

weight_missing_train_test = (
    train_weight_missing_by_store
    .join(
        test_weight_missing_by_store,
        lsuffix="_train",
        rsuffix="_test"
    )
)

display(weight_missing_train_test)

In [ ]:
# Inspect the existing weight-sales and weight-price summaries to determine whether observed weight has predictive structure.
print(" Weight vs Sales ")
print(weight_sales_profile.columns.tolist())
display(weight_sales_profile)

print("\n Weight vs Price ")
print(weight_price_corr.columns.tolist())
display(weight_price_corr)

print("\n Weight vs Category ")
print(weight_category_sales.columns.tolist())
display(weight_category_sales)

print("\n Weight Variation by Product ")
print(weight_variation_by_product.columns.tolist())
display(weight_variation_by_product.head(20))

In [ ]:
# Quantify within-product relative weight variation and test whether it increases with observation count.
weight_variation_analysis = weight_variation_by_product.copy()

weight_variation_analysis["weight_range_pct"] = (
    weight_variation_analysis["weight_range"]
    / weight_variation_analysis["median_weight"]
    * 100
)

weight_variation_summary = weight_variation_analysis[
    ["observations", "observed_values", "unique_weights", "median_weight", "weight_range", "weight_range_pct"]
].describe()

weight_variation_by_observations = (
    weight_variation_analysis
    .groupby("observations")
    .agg(
        products=("weight_range_pct", "size"),
        median_range_pct=("weight_range_pct", "median"),
        mean_range_pct=("weight_range_pct", "mean"),
        max_range_pct=("weight_range_pct", "max"),
        median_unique_weights=("unique_weights", "median")
    )
)

display(weight_variation_summary)
display(weight_variation_by_observations)

In [ ]:
# Compare sales between rows with observed and missing weight, first overall and then within stores that contain both states.
weight_missing_sales = train.copy()

weight_missing_sales["weight_missing"] = (
    weight_missing_sales["product_weight_kg"].isna()
)

overall_weight_missing_sales = (
    weight_missing_sales
    .groupby("weight_missing")["total_sales"]
    .agg(["count", "mean", "median", "std"])
)

within_store_weight_missing_sales = (
    weight_missing_sales
    .groupby(["store_code", "weight_missing"])["total_sales"]
    .agg(["count", "mean", "median", "std"])
)

display(overall_weight_missing_sales)

display(within_store_weight_missing_sales)

In [ ]:
# Test whether product-level typical weight has a stronger relationship with sales than row-level weight.
product_weight_signal = (
    train.groupby("product_code")
    .agg(
        product_median_weight=("product_weight_kg", "median"),
        product_mean_sales=("total_sales", "mean"),
        product_median_sales=("total_sales", "median"),
        observations=("total_sales", "size"),
    )
    .reset_index()
)

product_weight_signal_corr = (
    product_weight_signal[
        ["product_median_weight", "product_mean_sales", "product_median_sales"]
    ]
    .corr()
)

display(product_weight_signal.head())
display(product_weight_signal_corr)

In [ ]:
# Visualize the relationship between product-level typical weight and product-level sales.
import matplotlib.pyplot as plt

plot_weight_signal = product_weight_signal.dropna(
    subset=["product_median_weight", "product_mean_sales"]
)

plt.figure(figsize=(10, 6))

plt.scatter(
    plot_weight_signal["product_median_weight"],
    plot_weight_signal["product_mean_sales"],
    edgecolor = "black",
)

plt.xlabel("Product Median Weight (kg)")
plt.ylabel("Product Mean Sales")
plt.title("Product Median Weight vs Mean Sales")
plt.grid(alpha=0.4)

plt.tight_layout()
plt.savefig("../Gallery/product_median_weight_vs_sales.png", dpi=300, bbox_inches="tight")
plt.show()

In [242]:
# Shelf visibility: compact relationship check

visibility_profile = (
    train[["shelf_visibility", "total_sales"]]
    .describe()
    .T
)

visibility_corr = train[["shelf_visibility", "total_sales"]].corr().loc[
    "shelf_visibility", "total_sales"
]

visibility_bins = pd.qcut(
    train["shelf_visibility"],
    q=10,
    duplicates="drop"
)

visibility_sales_profile = (
    train.assign(visibility_bin=visibility_bins)
    .groupby("visibility_bin", observed=True)["total_sales"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

display(visibility_profile)
print(f"Pearson correlation: {visibility_corr:.4f}")
display(visibility_sales_profile)

,count,mean,std,min,25%,50%,75%,max
shelf_visibility,6818.0,0.065527,0.051566,0.0,0.0266,0.0532,0.0934,0.3201
total_sales,6818.0,2174.756574,1697.894956,32.7,834.7925,1790.8900,3092.5875,12996.8200


Pearson correlation: -0.1181


,visibility_bin,count,mean,median,std,min,max
0,"(-0.001, 0.0115]",683,2189.732577,1838.220,1595.636977,33.30,12996.82
1,"(0.0115, 0.022]",682,2345.404604,1950.685,1762.550278,80.42,12359.04
2,"(0.022, 0.0311]",682,2290.540557,2009.675,1616.827612,35.40,9803.38
3,"(0.0311, 0.041]",683,2366.624275,1932.320,1759.934920,60.92,10477.97
4,"(0.041, 0.0532]",683,2113.536003,1848.800,1581.991033,41.85,9768.51
5,"(0.0532, 0.0674]",679,2171.124080,1762.760,1706.692083,34.79,9999.51
6,"(0.0674, 0.0829]",680,2359.913426,1785.865,1915.006294,34.32,10130.90
7,"(0.0829, 0.106]",684,2249.370292,1873.005,1675.631412,54.93,9523.15
8,"(0.106, 0.14]",680,2067.929941,1727.510,1629.816931,32.70,7868.06
9,"(0.14, 0.32]",682,1593.171378,1051.540,1576.358414,34.56,9564.04


In [243]:
# Price × category: compact interaction check

price_category = (
    train.assign(product_category_clean=train["product_category"].str.strip().str.lower())
    .groupby("product_category_clean")
    .agg(
        n=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        price_sales_corr=("product_price", lambda x: x.corr(
            train.loc[x.index, "total_sales"]
        ))
    )
    .sort_values("mean_sales", ascending=False)
)

display(price_category)

,n,mean_price,mean_sales,median_sales,price_sales_corr
product_category_clean,,,,,
starchy foods,113,148.540708,2407.718584,2037.090,0.531799
household,740,149.347824,2292.464662,2023.745,0.519480
fruits and vegetables,1006,143.692266,2277.854672,1816.565,0.577114
meat,339,143.472035,2264.310383,1861.450,0.565322
seafood,52,139.830577,2262.524231,1929.040,0.570001
dairy,554,148.657527,2238.317076,1695.955,0.588729
snack foods,933,145.116817,2223.830568,1887.240,0.511334
canned,517,138.116228,2196.398743,1838.800,0.597569
hard drinks,169,138.355148,2189.723195,1842.880,0.600816


In [244]:
# Store × product category interaction: how much category mix varies by store

store_category = (
    train.groupby(["store_code", "product_category_clean"])["total_sales"]
    .agg(["count", "mean"])
    .reset_index()
)

store_category["store_category_share"] = (
    store_category["count"] /
    store_category.groupby("store_code")["count"].transform("sum")
)

display(
    store_category.sort_values(
        ["store_code", "store_category_share"],
        ascending=[True, False]
    ).head(30)
)

print(
    f"Unique store/category combinations: "
    f"{store_category.shape[0]}"
)

,store_code,product_category_clean,count,mean,store_category_share
6,STORE-7WS,fruits and vegetables,116,4021.409741,0.155080
13,STORE-7WS,snack foods,103,3575.135631,0.137701
9,STORE-7WS,household,82,3877.592927,0.109626
5,STORE-7WS,frozen foods,71,3597.773662,0.094920
0,STORE-7WS,baking goods,58,3277.200172,0.077540
3,STORE-7WS,canned,57,3629.159298,0.076203
4,STORE-7WS,dairy,52,3855.248269,0.069519
8,STORE-7WS,health and hygiene,47,3281.281277,0.062834
10,STORE-7WS,meat,43,4057.276512,0.057487
14,STORE-7WS,soft drinks,40,3427.897250,0.053476


Unique store/category combinations: 160


In [246]:
# Final modeling-readiness audit

audit = pd.DataFrame({
    "train_dtype": train.drop(columns="total_sales").dtypes.astype(str),
    "test_dtype": test.dtypes.astype(str),
    "train_nunique": train.drop(columns="total_sales").nunique(),
    "test_nunique": test.nunique(),
})

audit["test_values_unseen_in_train"] = [
    len(set(test[col].dropna().unique()) - set(train[col].dropna().unique()))
    for col in audit.index
]

display(audit)

print("\nTrain rows:", len(train))
print("Test rows:", len(test))
print("Train target missing:", train["total_sales"].isna().sum())
print("Duplicate train rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())

,train_dtype,test_dtype,train_nunique,test_nunique,test_values_unseen_in_train
id,object,object,6818,1705,1705
product_code,object,object,1555,1082,4
product_weight_kg,float64,float64,4675,1329,953
fat_content,object,object,2,2,0
shelf_visibility,float64,float64,1732,984,83
product_category,object,object,48,48,0
product_price,float64,float64,5818,1650,1182
store_code,object,object,10,10,0
store_age_years,int64,int64,9,9,0
store_size,object,object,3,3,0



Train rows: 6818
Test rows: 1705
Train target missing: 0
Duplicate train rows: 0
Duplicate test rows: 0


In [247]:
# Within-product price variation

product_price_variation = (
    train.groupby("product_code")["product_price"]
    .agg(
        observations="count",
        unique_prices="nunique",
        min_price="min",
        max_price="max",
        mean_price="mean",
        std_price="std"
    )
)

product_price_variation["price_range"] = (
    product_price_variation["max_price"]
    - product_price_variation["min_price"]
)

print("Products:", len(product_price_variation))
print(
    "Products with >1 observed price:",
    (product_price_variation["unique_prices"] > 1).sum()
)

display(
    product_price_variation
    .sort_values("price_range", ascending=False)
    .head(15)
)

Products: 1555
Products with >1 observed price: 1509


,observations,unique_prices,min_price,max_price,mean_price,std_price,price_range
product_code,,,,,,,
PRD-YDT7CU,7,7,253.93,270.87,261.800000,5.459020,16.94
PRD-44P39J,7,7,249.75,266.50,258.875714,6.674900,16.75
PRD-UWE98S,6,6,231.03,246.84,237.490000,6.491761,15.81
PRD-3I603D,8,8,254.03,269.49,265.127500,5.607048,15.46
PRD-B9OF3M,6,6,248.42,263.87,254.825000,6.680017,15.45
PRD-IF1HEB,7,7,226.86,242.13,236.504286,6.259433,15.27
PRD-MVFBO1,5,5,224.64,239.83,231.740000,5.680330,15.19
PRD-8VYI78,6,6,221.64,236.72,229.370000,5.826828,15.08
PRD-N5F62D,8,8,250.68,265.53,257.910000,5.264672,14.85


In [248]:
# Price-sales relationship by store

price_store = (
    train.groupby("store_code")
    .apply(
        lambda g: pd.Series({
            "n": len(g),
            "mean_price": g["product_price"].mean(),
            "mean_sales": g["total_sales"].mean(),
            "price_sales_corr": g["product_price"].corr(g["total_sales"])
        }),
        include_groups=False
    )
    .sort_values("mean_sales", ascending=False)
)

display(price_store)

,n,mean_price,mean_sales,price_sales_corr
store_code,,,,
STORE-7WS,748.0,139.438409,3660.182219,0.739044
STORE-9RG,752.0,144.001144,2479.168098,0.697980
STORE-89Z,728.0,140.135755,2365.926745,0.698582
STORE-OYG,744.0,141.508804,2365.833656,0.684577
STORE-AGY,748.0,139.974104,2254.890120,0.706163
STORE-YLW,754.0,141.642878,2253.950690,0.630516
STORE-DKU,736.0,139.637323,2177.136481,0.652420
STORE-HL7,742.0,140.185283,1971.661954,0.664365
STORE-T5G,427.0,137.880867,338.326651,0.578749


In [249]:
# Pearson vs Spearman: numeric relationships with sales

from scipy.stats import spearmanr

numeric_cols = [
    "product_weight_kg",
    "product_price",
    "shelf_visibility",
    "store_age_years"
]

numeric_relationships = []

for col in numeric_cols:
    tmp = train[[col, "total_sales"]].dropna()
    pearson = tmp[col].corr(tmp["total_sales"])
    spearman = spearmanr(tmp[col], tmp["total_sales"]).statistic

    numeric_relationships.append({
        "feature": col,
        "pearson": pearson,
        "spearman": spearman,
        "difference": abs(spearman - pearson)
    })

display(
    pd.DataFrame(numeric_relationships)
    .sort_values("difference", ascending=False)
)

,feature,pearson,spearman,difference
3,store_age_years,0.039443,-0.047506,0.086948
2,shelf_visibility,-0.118124,-0.105185,0.012939
1,product_price,0.569833,0.566437,0.003396
0,product_weight_kg,0.015532,0.017547,0.002015
